In [0]:
%sql
CREATE TABLE IF NOT EXISTS F1.Silver.ETL_batch_status (
    table_name  STRING  NOT NULL,
    batch_date  DATE    NOT NULL,
    status      STRING  NOT NULL,
    updated_at  TIMESTAMP
)
USING DELTA;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS F1.config.meta_table_config
(
    table_name      STRING      NOT NULL,   -- logical name, matches Workflow param
    pk_cols         STRING      NOT NULL,   -- comma-separated primary-key columns
    hash_cols       STRING      NOT NULL,   -- comma-separated columns for hash_diff
    partition_cols  STRING,                 -- comma-separated; NULL = no partitioning
    scd_type        STRING      NOT NULL,   -- SCD1 | SCD2 | SCD4
    dq_rule_group   STRING      NOT NULL,   -- FK → meta_dq_rules.rule_group
    staging_table   STRING      NOT NULL,   -- fully qualified Bronze/staging table
    target_table    STRING      NOT NULL,   -- fully qualified Silver target table
    history_table   STRING                  -- SCD4 history table (nullable)
)
USING DELTA;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS F1.config.meta_dq_rules
(
    rule_group      STRING      NOT NULL,   -- groups rules per table_config.dq_rule_group
    rule_id         STRING      NOT NULL,   -- unique rule identifier
    rule_type       STRING      NOT NULL,   -- NOT_NULL | RANGE | IN_LIST | EXPR
    column_name     STRING,                 -- target column (nullable for multi-col EXPR)
    spark_sql_expr  STRING      NOT NULL,   -- boolean expression; row passes when TRUE
    severity        STRING      NOT NULL,   -- ERROR | WARN
    reject_action   STRING      NOT NULL    -- FAIL_PIPELINE | QUARANTINE | IGNORE
)
USING DELTA;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS F1.config.dq_violations
(
    table_name      STRING,
    batch_date      DATE,
    rule_group      STRING,
    rule_id         STRING,
    rule_type       STRING,
    column_name     STRING,
    dq_severity     STRING,
    reject_action   STRING,
    pk_values       STRING,     -- JSON-serialised PK column values
    raw_row         STRING,     -- JSON-serialised full row
    logged_at       TIMESTAMP
)
USING DELTA;

In [0]:
%sql 
MERGE INTO F1.config.meta_table_config AS tgt
USING (
  SELECT * FROM VALUES
    ('circuits',     'circuit_id',            'circuit_ref,name,location,country,latitude,longitude,altitude',                             'country',     'SCD2', 'dq_circuits',     'F1.BRONZE.circuits',     'F1.Silver.circuits',     NULL),
    ('races',        'race_id',               'race_year,round,circuit_id,name,date,time',                                                  'race_year',   'SCD2', 'dq_races',        'F1.BRONZE.races',        'F1.Silver.races',        NULL),
    ('constructors', 'constructor_id',        'constructor_ref,name,nationality',                                                           'nationality', 'SCD2', 'dq_constructors', 'F1.BRONZE.constructors', 'F1.Silver.constructors', NULL),
    ('drivers',      'driver_id',             'driver_ref,number,code,name,dob,nationality',                                               'nationality', 'SCD2', 'dq_drivers',      'F1.BRONZE.drivers',      'F1.Silver.drivers',      NULL),
    ('results',      'result_id',             'race_id,driver_id,constructor_id,number,grid,position,points,laps,milliseconds,fastest_lap_speed', 'race_id', 'SCD4', 'dq_results', 'F1.BRONZE.results', 'F1.Silver.results', 'F1.Silver.results_history'),
    ('pit_stops',    'race_id,driver_id,stop','lap,time,duration,milliseconds',                                                             'race_id',     'SCD4', 'dq_pit_stops',    'F1.BRONZE.pit_stops',    'F1.Silver.pit_stops',    'F1.Silver.pit_stops_history'),
    ('lap_times',    'race_id,driver_id,lap', 'position,time,milliseconds',                                                                 'race_id',     'SCD4', 'dq_lap_times',    'F1.BRONZE.lap_times',    'F1.Silver.lap_times',    'F1.Silver.lap_times_history'),
    ('qualifying',   'qualify_id',            'race_id,driver_id,constructor_id,number,position,q1,q2,q3',                                 'race_id',     'SCD4', 'dq_qualifying',   'F1.BRONZE.qualifying',   'F1.Silver.qualifying',   'F1.Silver.qualifying_history')
    
  AS src(table_name, pk_cols, hash_cols, partition_cols, scd_type, dq_rule_group, staging_table, target_table, history_table)
) AS src
ON tgt.table_name = src.table_name

WHEN MATCHED THEN
UPDATE SET
  tgt.pk_cols        = src.pk_cols,
  tgt.hash_cols      = src.hash_cols,
  tgt.partition_cols = src.partition_cols,
  tgt.scd_type       = src.scd_type,
  tgt.dq_rule_group  = src.dq_rule_group,
  tgt.staging_table  = src.staging_table,
  tgt.target_table   = src.target_table,
  tgt.history_table  = src.history_table

WHEN NOT MATCHED THEN
INSERT (
  table_name,
  pk_cols,
  hash_cols,
  partition_cols,
  scd_type,
  dq_rule_group,
  staging_table,
  target_table,
  history_table
)
VALUES (
  src.table_name,
  src.pk_cols,
  src.hash_cols,
  src.partition_cols,
  src.scd_type,
  src.dq_rule_group,
  src.staging_table,
  src.target_table,
  src.history_table
);

In [0]:
%sql 
MERGE INTO F1.config.meta_dq_rules AS tgt
USING (
  SELECT * FROM VALUES
    -- circuits
    ('dq_circuits','circuits_r01','NOT_NULL','circuit_id', 'circuit_id IS NOT NULL',                                           'ERROR','FAIL_PIPELINE'),
    ('dq_circuits','circuits_r02','NOT_NULL','name',       'name IS NOT NULL AND TRIM(name) <> ""',                            'ERROR','FAIL_PIPELINE'),
    ('dq_circuits','circuits_r03','RANGE',  'latitude',   'latitude BETWEEN -90.0 AND 90.0',                                  'ERROR','QUARANTINE'),
    ('dq_circuits','circuits_r04','RANGE',  'longitude',  'longitude BETWEEN -180.0 AND 180.0',                               'ERROR','QUARANTINE'),
    ('dq_circuits','circuits_r05','NOT_NULL','country',    'country IS NOT NULL',                                              'WARN', 'IGNORE'),
    -- races
    ('dq_races','races_r01','NOT_NULL','race_id',   'race_id IS NOT NULL',                                                     'ERROR','FAIL_PIPELINE'),
    ('dq_races','races_r02','RANGE',  'race_year', 'race_year BETWEEN 1950 AND 2100',                                         'ERROR','QUARANTINE'),
    ('dq_races','races_r03','NOT_NULL','circuit_id','circuit_id IS NOT NULL',                                                  'ERROR','FAIL_PIPELINE'),
    ('dq_races','races_r04','NOT_NULL','name',      'name IS NOT NULL AND TRIM(name) <> ""',                                   'WARN', 'IGNORE'),
    -- constructors
    ('dq_constructors','cons_r01','NOT_NULL','constructor_id', 'constructor_id IS NOT NULL',                                   'ERROR','FAIL_PIPELINE'),
    ('dq_constructors','cons_r02','NOT_NULL','name',           'name IS NOT NULL AND TRIM(name) <> ""',                        'ERROR','FAIL_PIPELINE'),
    ('dq_constructors','cons_r03','NOT_NULL','nationality',    'nationality IS NOT NULL',                                      'WARN', 'IGNORE'),
    ('dq_constructors','cons_r04','NOT_NULL','constructor_ref','constructor_ref IS NOT NULL',                                   'ERROR','QUARANTINE'),
    -- drivers
    ('dq_drivers','drivers_r01','NOT_NULL','driver_id',   'driver_id IS NOT NULL',                                            'ERROR','FAIL_PIPELINE'),
    ('dq_drivers','drivers_r02','NOT_NULL','name',        'name IS NOT NULL AND TRIM(name) <> ""',                            'ERROR','FAIL_PIPELINE'),
    ('dq_drivers','drivers_r03','EXPR',    'dob',         'dob IS NULL OR (dob >= DATE("1900-01-01") AND dob <= current_date())', 'ERROR','QUARANTINE'),
    ('dq_drivers','drivers_r04','NOT_NULL','nationality', 'nationality IS NOT NULL',                                          'WARN', 'IGNORE'),
    -- results
    ('dq_results','results_r01','NOT_NULL','result_id',         'result_id IS NOT NULL',                                      'ERROR','FAIL_PIPELINE'),
    ('dq_results','results_r02','NOT_NULL','race_id',           'race_id IS NOT NULL',                                        'ERROR','FAIL_PIPELINE'),
    ('dq_results','results_r03','NOT_NULL','driver_id',         'driver_id IS NOT NULL',                                      'ERROR','FAIL_PIPELINE'),
    ('dq_results','results_r04','RANGE',  'points',            'points IS NULL OR points >= 0',                              'WARN', 'IGNORE'),
    ('dq_results','results_r05','RANGE',  'fastest_lap_speed', 'fastest_lap_speed IS NULL OR fastest_lap_speed > 0',         'WARN', 'IGNORE'),
    -- pit_stops
    ('dq_pit_stops','ps_r01','NOT_NULL','race_id',    'race_id IS NOT NULL',                                                  'ERROR','FAIL_PIPELINE'),
    ('dq_pit_stops','ps_r02','NOT_NULL','driver_id',  'driver_id IS NOT NULL',                                                'ERROR','FAIL_PIPELINE'),
    ('dq_pit_stops','ps_r03','RANGE',  'lap',        'lap >= 1',                                                             'ERROR','QUARANTINE'),
    ('dq_pit_stops','ps_r04','RANGE',  'milliseconds','milliseconds > 0',                                                    'WARN', 'IGNORE'),
    -- lap_times
    ('dq_lap_times','lt_r01','NOT_NULL','race_id',    'race_id IS NOT NULL',                                                  'ERROR','FAIL_PIPELINE'),
    ('dq_lap_times','lt_r02','NOT_NULL','driver_id',  'driver_id IS NOT NULL',                                                'ERROR','FAIL_PIPELINE'),
    ('dq_lap_times','lt_r03','RANGE',  'lap',        'lap >= 1',                                                             'ERROR','QUARANTINE'),
    ('dq_lap_times','lt_r04','RANGE',  'milliseconds','milliseconds > 0',                                                    'WARN', 'IGNORE'),
    ('dq_lap_times','lt_r05','RANGE',  'position',   'position >= 1',                                                        'WARN', 'IGNORE'),
    -- qualifying
    ('dq_qualifying','qual_r01','NOT_NULL','qualify_id','qualify_id IS NOT NULL',                                             'ERROR','FAIL_PIPELINE'),
    ('dq_qualifying','qual_r02','NOT_NULL','race_id',   'race_id IS NOT NULL',                                                'ERROR','FAIL_PIPELINE'),
    ('dq_qualifying','qual_r03','NOT_NULL','driver_id', 'driver_id IS NOT NULL',                                              'ERROR','FAIL_PIPELINE'),
    ('dq_qualifying','qual_r04','RANGE',  'position',  'position >= 1',                                                      'WARN', 'IGNORE')
  AS src(rule_group,rule_id,rule_type,column_name,spark_sql_expr,severity,reject_action)
) AS src
ON tgt.rule_group = src.rule_group AND tgt.rule_id = src.rule_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
%sql
--F1.Silver
DROP TABLE IF EXISTS F1.Silver.circuits;
CREATE TABLE IF NOT EXISTS F1.Silver.circuits
(
	circuit_id INT,
	circuit_ref STRING,
	name STRING,
	location STRING,
	country STRING,
	latitude DOUBLE,
	longitude DOUBLE,
	altitude INT,
	url STRING,
	ingestion_date timestamp,
	data_source STRING,
	file_date STRING,
	row_hash STRING,
	eff_Start_date date,
	eff_End_date date,
	is_current int,
	created_at timestamp,
	updated_at timestamp
)
LOCATION 'abfss://dev@f1storage02.dfs.core.windows.net/Silver/circuits';

DROP TABLE IF EXISTS F1.Silver.races;
CREATE TABLE IF NOT EXISTS F1.Silver.races
(   race_id INT,
	race_year INT,
	round INT,
	circuit_id INT,
	name STRING,
	date DATE,
	time STRING,
	url STRING,
	ingestion_date timestamp,
	data_source STRING,
	file_date STRING,
	row_hash STRING,
	eff_Start_date date,
	eff_End_date date,
	is_current int,
	created_at timestamp,
	updated_at timestamp
)
LOCATION 'abfss://dev@f1storage02.dfs.core.windows.net/Silver/races';


DROP TABLE IF EXISTS F1.Silver.constructors;
CREATE TABLE IF NOT EXISTS F1.Silver.constructors(
constructor_id int,
constructor_ref STRING,
name STRING,
nationality STRING,
url STRING,
ingestion_date timestamp,
data_source STRING,
file_date STRING,
row_hash STRING,
eff_Start_date date,
eff_End_date date,
is_current int,
created_at timestamp,
updated_at timestamp
)
LOCATION 'abfss://dev@f1storage02.dfs.core.windows.net/Silver/constructors';

DROP TABLE IF EXISTS F1.Silver.drivers;
CREATE TABLE IF NOT EXISTS F1.Silver.drivers(
driver_id INT,
driver_ref STRING,
number INT,
code STRING,
name STRING,
dob DATE,
nationality STRING,
url STRING,
ingestion_date timestamp,
data_source STRING,
file_date STRING,
row_hash STRING,
eff_Start_date date,
eff_End_date date,
is_current int,
created_at timestamp,
updated_at timestamp
)
LOCATION 'abfss://dev@f1storage02.dfs.core.windows.net/Silver/drivers';

DROP TABLE IF EXISTS F1.Silver.results;
CREATE TABLE IF NOT EXISTS F1.Silver.results(
result_id INT,
race_id INT,
driver_id INT,
constructor_Id INT,
number INT,grid INT,
position STRING,
position_Text INT,
position_Order INT,
points INT,
laps INT,
time STRING,
milliseconds INT,
fastest_lap INT,
rank INT,
fastest_Lap_Time STRING,
fastest_Lap_Speed FLOAT,
status_Id STRING,
ingestion_date timestamp,
data_source STRING,
file_date STRING,
row_hash STRING,
eff_Start_date date,
eff_End_date date,
is_current int,
created_at timestamp,
updated_at timestamp
)
LOCATION 'abfss://dev@f1storage02.dfs.core.windows.net/Silver/results';


DROP TABLE IF EXISTS F1.Silver.pit_stops;
CREATE TABLE IF NOT EXISTS F1.Silver.pit_stops(
driver_Id INT,
duration STRING,
lap INT,
milliseconds INT,
race_Id INT,
stop INT,
time STRING,
ingestion_date timestamp,
data_source STRING,
file_date STRING,
row_hash STRING,
eff_Start_date date,
eff_End_date date,
is_current int,
created_at timestamp,
updated_at timestamp
)
LOCATION 'abfss://dev@f1storage02.dfs.core.windows.net/Silver/pit_stops';

DROP TABLE IF EXISTS F1.Silver.lap_times;
CREATE TABLE IF NOT EXISTS F1.Silver.lap_times(
race_Id INT,
driver_Id INT,
lap INT,
position INT,
time STRING,
milliseconds INT,
ingestion_date timestamp,
data_source STRING,
file_date STRING,
row_hash STRING,
eff_Start_date date,
eff_End_date date,
is_current int,
created_at timestamp,
updated_at timestamp
)
LOCATION 'abfss://dev@f1storage02.dfs.core.windows.net/Silver/lap_times';

DROP TABLE IF EXISTS F1.Silver.qualifying;
CREATE TABLE IF NOT EXISTS F1.Silver.qualifying(
constructor_Id INT,
driver_Id INT,
number INT,
position INT,
q1 STRING,
q2 STRING,
q3 STRING,
qualify_Id INT,
race_Id INT,
ingestion_date timestamp,
data_source STRING,
file_date STRING,
row_hash STRING,
eff_Start_date date,
eff_End_date date,
is_current int,
created_at timestamp,
updated_at timestamp
)
LOCATION 'abfss://dev@f1storage02.dfs.core.windows.net/Silver/qualifying';


CREATE TABLE IF NOT EXISTS F1.Silver.ETL_batch_status
(
    table_name  STRING,
    batch_date  DATE,
    status      STRING,
    updated_at  TIMESTAMP
)
USING DELTA;

INSERT INTO F1.Silver.ETL_batch_status VALUES
('circuits',     '2021-04-18', 'SUCCESS', current_timestamp()),
('races',        '2021-04-18', 'SUCCESS', current_timestamp()),
('constructors', '2021-04-18', 'SUCCESS', current_timestamp()),
('drivers',      '2021-04-18', 'SUCCESS', current_timestamp()),
('results',      '2021-04-18', 'SUCCESS', current_timestamp()),
('lap_times',    '2021-04-18', 'SUCCESS', current_timestamp()),
('pit_stops',    '2021-04-18', 'SUCCESS', current_timestamp()),
('qualifying',   '2021-04-18', 'SUCCESS', current_timestamp());

-- F1 Silver Layer – SCD4 History Tables
-- Run once to create history tables for each FACT table.
-- These mirror the FACT table schema plus an `archived_at` audit column.

-- ── results_history ──────────────────────────────────────────
CREATE TABLE IF NOT EXISTS F1.Silver.results_history
(
    result_id         INT,
    race_id           INT,
    driver_id         INT,
    constructor_id    INT,
    number            INT,
    grid              INT,
    position          STRING,
    position_text     INT,
    position_order    INT,
    points            INT,
    laps              INT,
    time              STRING,
    milliseconds      INT,
    fastest_lap       INT,
    rank              INT,
    fastest_lap_time  STRING,
    fastest_lap_speed FLOAT,
    status_id         STRING,
    ingestion_date    TIMESTAMP,
    data_source       STRING,
    file_date         STRING,
    row_hash          STRING,
    eff_start_date    DATE,
    eff_end_date      DATE,
    is_current        INT,
    created_at        TIMESTAMP,
    updated_at        TIMESTAMP,
    archived_at       TIMESTAMP   -- SCD4 audit column
)
USING DELTA
LOCATION 'abfss://dev@f1storage02.dfs.core.windows.net/Silver/results_history';

-- ── lap_times_history ────────────────────────────────────────
CREATE TABLE IF NOT EXISTS F1.Silver.lap_times_history
(
    race_id        INT,
    driver_id      INT,
    lap            INT,
    position       INT,
    time           STRING,
    milliseconds   INT,
    ingestion_date TIMESTAMP,
    data_source    STRING,
    file_date      STRING,
    row_hash       STRING,
    eff_start_date DATE,
    eff_end_date   DATE,
    is_current     INT,
    created_at     TIMESTAMP,
    updated_at     TIMESTAMP,
    archived_at    TIMESTAMP
)
USING DELTA
LOCATION 'abfss://dev@f1storage02.dfs.core.windows.net/Silver/lap_times_history';

-- ── pit_stops_history ────────────────────────────────────────
CREATE TABLE IF NOT EXISTS F1.Silver.pit_stops_history
(
    driver_id      INT,
    race_id        INT,
    stop           INT,
    lap            INT,
    time           STRING,
    duration       STRING,
    milliseconds   INT,
    ingestion_date TIMESTAMP,
    data_source    STRING,
    file_date      STRING,
    row_hash       STRING,
    eff_start_date DATE,
    eff_end_date   DATE,
    is_current     INT,
    created_at     TIMESTAMP,
    updated_at     TIMESTAMP,
    archived_at    TIMESTAMP
)
USING DELTA
LOCATION 'abfss://dev@f1storage02.dfs.core.windows.net/Silver/pit_stops_history';

-- ── qualifying_history ───────────────────────────────────────
CREATE TABLE IF NOT EXISTS F1.Silver.qualifying_history
(
    qualify_id     INT,
    race_id        INT,
    driver_id      INT,
    constructor_id INT,
    number         INT,
    position       INT,
    q1             STRING,
    q2             STRING,
    q3             STRING,
    ingestion_date TIMESTAMP,
    data_source    STRING,
    file_date      STRING,
    row_hash       STRING,
    eff_start_date DATE,
    eff_end_date   DATE,
    is_current     INT,
    created_at     TIMESTAMP,
    updated_at     TIMESTAMP,
    archived_at    TIMESTAMP
)
USING DELTA
LOCATION 'abfss://dev@f1storage02.dfs.core.windows.net/Silver/qualifying_history';


In [0]:
%sql
Select * from F1.Config.meta_table_config; --WHERE table_name='circuits';